# NAMD-FEP — notebook 1 · visualise the protein–ligand complex

The starting point of this workflow is a **docked pose or a crystal structure** —
for example the output of the docking notebooks on
[quantaosun/labodock_binder](https://github.com/quantaosun/labodock_binder).
Before anything is built or simulated, **look at the complex and inventory the file.**
Every downstream decision (which chain, which ligand, which atoms differ) is made here.

What this notebook does:

1. get a complex — fetch a PDB entry, upload your own, or point at a local file;
2. **`rbfe-prep report`** — a read-only inventory: chains, HETATM species, numbering
   gaps, altlocs, modified residues;
3. an interactive 3D view of the complex with the ligand highlighted;
4. record the three choices the next notebook needs: the protein chain, the ligand
   residue name, and the tutorial workspace.

> Nothing in this notebook modifies a file. It is safe to re-run.

## 1.1 Setup

The same bootstrap as notebook 00: locate the repository, put `bin/` on PATH, and
create the tutorial workspace. Each notebook is self-contained — run this cell first.

In [ ]:
import os, sys
from pathlib import Path

REPO = Path.cwd().resolve()
if not (REPO / "bin" / "rbfe").exists():        # running from notebooks/
    REPO = REPO.parent
os.environ["PATH"] = str(REPO / "bin") + os.pathsep + os.environ["PATH"]
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

WORK = REPO / "_tutorial"
(WORK / "inputs").mkdir(parents=True, exist_ok=True)
print("repo root         :", REPO)
print("tutorial workspace:", WORK)

## 1.2 Choose the complex

Three ways in — pick **one** and run only its cell.

**A. Fetch a PDB entry.** The worked example below is **6I5I** (CLK1 kinase with the
ligand H3E), the same system the repository's published ΔΔG was computed on, so the
tutorial's outputs can be checked against real results. **3HTB** (the `systems/3htb`
worked example for `rbfe-prep`) works too.

In [ ]:
import subprocess

PDB_ID = "6I5I"                       # <-- change me, e.g. "3HTB"
COMPLEX_PDB = WORK / f"{PDB_ID}.pdb"

if not COMPLEX_PDB.exists():
    print("fetching needs network access; if this session has none, download\n"
          "https://files.rcsb.org/download/" + PDB_ID + ".pdb by hand and use\n"
          "option B or C below.")
    r = subprocess.run(["rbfe-prep", "fetch", PDB_ID, "-o", str(COMPLEX_PDB)],
                       capture_output=True, text=True)
    print(r.stdout, r.stderr)
else:
    print(f"already here: {COMPLEX_PDB}")
if COMPLEX_PDB.exists():
    print(f"{COMPLEX_PDB} ({sum(1 for _ in COMPLEX_PDB.open())} lines)")

**B. Upload your own complex** (a docked pose from the labodock notebooks, a local
crystal structure, ...). Run the cell, pick the file, and it lands in the workspace.
Requires `ipywidgets` (present on any JupyterLab/Binder image).

In [ ]:
import shutil

try:
    import ipywidgets as widgets
except ImportError:
    widgets = None
    print("ipywidgets is not available in this kernel -- use option A or C.")

if widgets is not None:
    up = widgets.FileUpload(accept=".pdb", multiple=False,
                            description="Upload .pdb")
    display(up)

    def _stash(_):
        global COMPLEX_PDB
        if not up.value:
            return
        item = up.value[0] if isinstance(up.value, tuple) else list(up.value.values())[0]
        name, content = item["name"], item["content"]
        COMPLEX_PDB = WORK / Path(name).name
        COMPLEX_PDB.write_bytes(content)
        print(f"saved {name} -> {COMPLEX_PDB}")

    up.observe(_stash, names="value")

**C. Point at a file that is already here.** Useful on a local Jupyter, or for the
frozen example that ships with the repository:

In [ ]:
# Option C -- uncomment ONE of these:
# COMPLEX_PDB = REPO / "6I5I_DUAL_FEP" / "inputs" / "protein.pdb"   # frozen 6I5I protein only
# COMPLEX_PDB = Path("/path/to/my/complex.pdb")                      # any local file

assert COMPLEX_PDB.exists(), f"{COMPLEX_PDB} does not exist"
print("using", COMPLEX_PDB)

## 1.3 Inventory the file — `rbfe-prep report`

Read-only. This is the single most important cell in the notebook: it answers the
questions that decide everything else.

* **chains** — which one holds your ligand? `--chain` in notebook 02 takes one.
* **HETATM species** — the ligand's residue name (e.g. `H3E` for 6I5I, `JZ4` for 3HTB),
  plus water/ions/crystallisation additives, which `strip` removes by default.
* **numbering gaps** — a disordered gap is fine; `rbfe build` splits segments there.
* **altlocs** — `strip` keeps the blank/major conformer.
* **⚠ modified residues** — MSE, SEP, TPO & friends **will not build** in psfgen and
  must be handled before step ②.

In [ ]:
import subprocess

r = subprocess.run(["rbfe-prep", "report", str(COMPLEX_PDB)],
                   capture_output=True, text=True)
print(r.stdout)
if r.returncode != 0:
    raise RuntimeError(r.stderr)

## 1.4 Visualise the complex in 3D

Set the two choices the report just informed, then look at the structure. Rotate with
the mouse; the ligand is drawn as ball-and-stick, the protein as a cartoon.

In [ ]:
CHAIN   = "A"     # <-- the chain to keep (from the report above; '' keeps all)
LIG_RES = "H3E"   # <-- the ligand's HETATM residue name (e.g. "JZ4" for 3HTB)

import nglview as nv

view = nv.show_structure_file(str(COMPLEX_PDB))
view.clear_representations()
view.add_representation("cartoon", selection="protein", color="lightgrey")
view.add_representation("ball+stick", selection=f"{LIG_RES}", color="element")
view.add_representation("licorice", selection="hetero and not water and not " + LIG_RES)
view.add_representation("spacefill", selection=f"({LIG_RES}) and not _H",
                        opacity=0.25)
view.center(selection=LIG_RES)
print(f"{COMPLEX_PDB.name}: protein as cartoon, {LIG_RES} as ball-and-stick")
view

## 1.5 Record the choices

These three values are everything notebook 02 needs. Write them down here so the next
notebook can pick them up without re-asking.

In [ ]:
import json

choices = {"complex_pdb": str(COMPLEX_PDB), "chain": CHAIN, "ligand_resname": LIG_RES}
(WORK / "choices.json").write_text(json.dumps(choices, indent=2) + "\n")
print(json.dumps(choices, indent=2))
print(f"\nsaved to {WORK / 'choices.json'} -- notebook 02 reads this.")
print("Next: 02_prepare_and_sanitise.ipynb")